# 📈 Portfolio Theory & Optimisation — Week 1 Assignment
### UAPOML Summer Project | IIT Kanpur 2025
---
**Submitted Solution — All 15 Questions (Theoretical + Coding)**

> **Instructions recap:** Theoretical answers show full working. Code is vectorised (no loops where NumPy suffices). All outputs are reproducible.


## 🔧 Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

np.random.seed(42)          # reproducibility for all random operations
plt.rcParams.update({
    'figure.dpi'      : 120,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'font.family'     : 'DejaVu Sans',
})
print("All imports successful ✓")


---
# §1 Theoretical Questions (Q1–Q9)


## Question 1 [2 marks] — Simple Returns, Log Returns, Approximation

**Given monthly prices:** P₀=100, P₁=108, P₂=103, P₃=115, P₄=110

### (a) Simple Monthly Returns  Rₜ = (Pₜ − Pₜ₋₁) / Pₜ₋₁


In [ ]:
prices = np.array([100, 108, 103, 115, 110])

# (a) Simple returns
R = np.diff(prices) / prices[:-1]
print("=== (a) Simple Returns Rt ===")
for i, r in enumerate(R, 1):
    print(f"  R{i} = ({prices[i]} - {prices[i-1]}) / {prices[i-1]} = {r:.6f}  ({r*100:.4f}%)")


### (b) Log Returns  rₜ = ln(Pₜ / Pₜ₋₁)


In [ ]:
# (b) Log returns
r = np.log(prices[1:] / prices[:-1])
print("=== (b) Log Returns rt ===")
for i, lr in enumerate(r, 1):
    print(f"  r{i} = ln({prices[i]}/{prices[i-1]}) = {lr:.6f}  ({lr*100:.4f}%)")


### (c) Approximation rₜ ≈ Rₜ for small returns

**Theory:** Using Taylor expansion: ln(1+x) ≈ x − x²/2 + ...  
So rₜ = ln(Pₜ/Pₜ₋₁) = ln(1 + Rₜ) ≈ Rₜ when Rₜ is small.  
The approximation error ≈ −Rₜ²/2, so **larger |Rₜ| → worse approximation**.


In [ ]:
# (c) Verify approximation and find least accurate month
print("=== (c) Approximation Check ===")
print(f"{'Month':>6} {'Rt':>10} {'rt':>10} {'|Rt-rt|':>12} {'Approx error (%)':>18}")
print("-" * 60)
for i in range(4):
    diff = abs(R[i] - r[i])
    print(f"  M{i+1}   {R[i]:>10.6f} {r[i]:>10.6f} {diff:>12.8f} {diff*100:>16.6f}%")

worst = np.argmax(np.abs(R - r)) + 1
print(f"\n→ Approximation LEAST accurate at Month {worst}")
print(f"  Reason: Month {worst} has the largest |Rt| = {abs(R[worst-1])*100:.2f}%,")
print(f"  so the higher-order term Rt²/2 ≈ {R[worst-1]**2/2*100:.4f}% is largest.")


## Question 2 [1 mark] — Portfolio Expected Return

**Given:** E[R₁]=15%, E[R₂]=7%, w₁=0.4, w₂=0.6

### (a) Linearity of Expectation

**Formal statement:**  
For any random variables X, Y and constants a, b:  
$$E[aX + bY] = a\,E[X] + b\,E[Y]$$

**Derivation:**  
$$E[R_p] = E[w_1 R_1 + w_2 R_2] = w_1\,E[R_1] + w_2\,E[R_2]$$

$$E[R_p] = 0.4 \times 0.15 + 0.6 \times 0.07$$

### (b) Does correlation change E[Rp]?

**No.** The linearity of expectation holds for **any** joint distribution of R₁ and R₂ — it does **not** require independence. Correlation affects the **variance** (risk) of the portfolio, not its expected return.


In [ ]:
# Q2 Numerical verification
E_R1, E_R2 = 0.15, 0.07
w1, w2     = 0.40, 0.60
E_Rp = w1 * E_R1 + w2 * E_R2
print(f"E[Rp] = {w1}×{E_R1} + {w2}×{E_R2} = {E_Rp:.4f}  ({E_Rp*100:.2f}%)")
print("Correlation does NOT affect E[Rp] — only σp² changes with ρ.")


## Question 3 [2 marks] — Derivation of Two-Asset Portfolio Variance

**Starting identity:** Var(X + Y) = Var(X) + Var(Y) + 2 Cov(X,Y)

**Step 1:** Write portfolio return as a sum:
$$R_p = w_1 R_1 + w_2 R_2$$

**Step 2:** Apply variance operator to a linear combination. For constants a, b:
$$\text{Var}(aX + bY) = a^2\,\text{Var}(X) + b^2\,\text{Var}(Y) + 2ab\,\text{Cov}(X,Y)$$

*Proof of this:*  
$$\text{Var}(aX+bY) = E[(aX+bY - aμ_X - bμ_Y)^2]$$
$$= E[(a(X-μ_X)+b(Y-μ_Y))^2]$$
$$= a^2 E[(X-μ_X)^2] + b^2 E[(Y-μ_Y)^2] + 2ab\,E[(X-μ_X)(Y-μ_Y)]$$
$$= a^2\,\text{Var}(X) + b^2\,\text{Var}(Y) + 2ab\,\text{Cov}(X,Y)$$

**Step 3:** Apply to portfolio with a=w₁, b=w₂, X=R₁, Y=R₂:
$$\sigma_p^2 = w_1^2\,\sigma_1^2 + w_2^2\,\sigma_2^2 + 2w_1 w_2\,\text{Cov}(R_1,R_2)$$

**Step 4:** Substitute Cov(R₁,R₂) = ρ₁₂ σ₁ σ₂:
$$\boxed{\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1 w_2\,\rho_{12}\,\sigma_1\sigma_2}$$


## Question 4 [1 mark] — Portfolio Risk at Different Correlations

**Given:** μ_A=18%, σ_A=25%, μ_B=8%, σ_B=12%, w_A=w_B=0.5

$$\sigma_p^2 = (0.5)^2(0.25)^2 + (0.5)^2(0.12)^2 + 2(0.5)(0.5)\rho\,(0.25)(0.12)$$


In [ ]:
# Q4 — Portfolio risk at ρ = +1, 0, -1
sig_A, sig_B = 0.25, 0.12
wA, wB = 0.5, 0.5

def port_sigma(rho, wA=0.5, wB=0.5, sA=0.25, sB=0.12):
    var = wA**2*sA**2 + wB**2*sB**2 + 2*wA*wB*rho*sA*sB
    return np.sqrt(var)

for rho in [1, 0, -1]:
    sp = port_sigma(rho)
    print(f"  ρ = {rho:+d}  →  σp = {sp:.6f}  ({sp*100:.4f}%)")

print()
# (b) Weighted average
w_avg = wA*sig_A + wB*sig_B
print(f"Weighted average of individual risks: wA·σA + wB·σB = {w_avg:.6f} ({w_avg*100:.4f}%)")
print(f"σp at ρ=+1 = {port_sigma(1):.6f}  →  Equal to weighted average? {np.isclose(port_sigma(1), w_avg)}")


### (b) When does σp equal the weighted average?

**When ρ = +1.**  Proof:  
$$\sigma_p^2 = w_A^2\sigma_A^2 + w_B^2\sigma_B^2 + 2w_A w_B \sigma_A\sigma_B$$
$$= (w_A\sigma_A + w_B\sigma_B)^2$$
$$\Rightarrow \sigma_p = w_A\sigma_A + w_B\sigma_B \quad \checkmark$$
This is simply the binomial square identity — perfect positive correlation means no diversification benefit.

### (c) Weight w*_A that drives σp to zero when ρ = −1

When ρ = −1:
$$\sigma_p^2 = (w_A\sigma_A - w_B\sigma_B)^2 = (w_A\sigma_A - (1-w_A)\sigma_B)^2$$

Setting σp = 0:
$$w_A\sigma_A = (1-w_A)\sigma_B \Rightarrow w_A(\sigma_A + \sigma_B) = \sigma_B$$
$$\boxed{w_A^* = \frac{\sigma_B}{\sigma_A + \sigma_B}}$$


In [ ]:
# (c) Zero-risk weight
wA_star = sig_B / (sig_A + sig_B)
wB_star = 1 - wA_star
print(f"w*_A = σ_B / (σ_A + σ_B) = {sig_B} / {sig_A+sig_B} = {wA_star:.6f} ({wA_star*100:.4f}%)")
print(f"w*_B = {wB_star:.6f} ({wB_star*100:.4f}%)")
sp_check = port_sigma(-1, wA_star, wB_star)
print(f"Verification: σp at ρ=-1, w*_A={wA_star:.4f} → {sp_check:.2e}  (≈ 0 ✓)")


## Question 5 [1 mark] — Why Combining Assets with ρ < 1 Reduces Risk

From the portfolio variance formula:
$$\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1 w_2\,\rho_{12}\,\sigma_1\sigma_2$$

The weighted-average variance is $(w_1\sigma_1 + w_2\sigma_2)^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\sigma_1\sigma_2$.

The **difference** between portfolio variance and weighted-average variance squared is:
$$2w_1 w_2\,\sigma_1\sigma_2\,(\rho_{12} - 1)$$

When **ρ₁₂ < 1**, this term is **strictly negative**, so:
$$\sigma_p^2 < (w_1\sigma_1 + w_2\sigma_2)^2 \Rightarrow \sigma_p < w_1\sigma_1 + w_2\sigma_2$$

Intuitively, when ρ < 1 the assets do not move in perfect lockstep. Gains in one asset partially offset losses in the other, reducing overall portfolio volatility. The lower ρ is, the greater this cancellation effect — the **diversification benefit**.


## Question 6 [1 mark] — 3-Asset Covariance Matrix

**Formula:** Σᵢⱼ = ρᵢⱼ σᵢ σⱼ


In [ ]:
# Q6 — Covariance matrix construction
mu  = np.array([0.15, 0.08, 0.05])
sig = np.array([0.25, 0.12, 0.04])

rho = np.array([
    [1.0, 0.4, 0.1],
    [0.4, 1.0, 0.2],
    [0.1, 0.2, 1.0]
])

# Σ = diag(σ) @ rho @ diag(σ)   — elegant vectorised construction
Sigma = np.diag(sig) @ rho @ np.diag(sig)

print("=== 3×3 Covariance Matrix Σ ===")
assets = ['Asset 1', 'Asset 2', 'Asset 3']
print(f"{'':>10}", end='')
for a in assets: print(f"{a:>12}", end='')
print()
for i, row in enumerate(Sigma):
    print(f"{assets[i]:>10}", end='')
    for v in row:
        print(f"{v:>12.6f}", end='')
    print()

print()
# (b) Symmetry check
is_sym = np.allclose(Sigma, Sigma.T)
print(f"(b) Σ is symmetric: {is_sym}  (Σ == Σᵀ element-wise ✓)")

# Spot-check one entry
print(f"\nSpot-check: Σ₁₂ = ρ₁₂·σ₁·σ₂ = {rho[0,1]}×{sig[0]}×{sig[1]} = {rho[0,1]*sig[0]*sig[1]:.6f}")
print(f"            Σ[0,1] from matrix  = {Sigma[0,1]:.6f}  ✓")


## Question 7 [1 mark] — Sharpe Ratios

**Sharpe Ratio:** S = (E[Rp] − Rf) / σp


In [ ]:
# Q7 — Sharpe Ratios
Rf = 0.04
portfolios = {
    'X': {'E_Rp': 0.14, 'sig_p': 0.22},
    'Y': {'E_Rp': 0.09, 'sig_p': 0.10},
    'Z': {'E_Rp': 0.20, 'sig_p': 0.35},
}

print("=== (a) Sharpe Ratios ===")
print(f"{'Portfolio':>12} {'E[Rp]':>8} {'σp':>8} {'Sharpe':>10}")
print("-" * 42)
sharpes = {}
for name, p in portfolios.items():
    S = (p['E_Rp'] - Rf) / p['sig_p']
    sharpes[name] = S
    print(f"{name:>12} {p['E_Rp']*100:>7.1f}% {p['sig_p']*100:>7.1f}% {S:>10.4f}")

ranked = sorted(sharpes, key=sharpes.get, reverse=True)
print(f"\n(b) Ranking by Sharpe Ratio: {' > '.join(ranked)}")
print(f"    A rational mean-variance investor prefers Portfolio {ranked[0]}")
print(f"    because it offers the highest return per unit of risk taken.")
print()
print("(c) Portfolio Z has the highest absolute return (20%) but its")
print("    σp = 35% means it has the LOWEST Sharpe Ratio (0.457).")
print("    Rational mean-variance investors maximise risk-ADJUSTED return.")
print("    Portfolio Y delivers 9% return with only 10% risk → 0.50 Sharpe.")
print("    In mean-variance framework, Y is on a higher indifference curve")
print("    than Z for any investor whose utility penalises variance.")


## Question 8 [1 mark] — GMV Portfolio, Efficient Frontier, CML

### (a) Global Minimum Variance (GMV) Portfolio

The **GMV portfolio** is the combination of risky assets that yields the **lowest possible variance** (σ²) across all feasible portfolios, with no constraint on expected return.

It is the **leftmost point** on the minimum variance frontier because the x-axis represents σp: all other feasible portfolios lie to the right (higher σ). No portfolio can have lower σ than the GMV portfolio — by definition, it is the variance-minimising solution.

Mathematically it solves:  
$$\min_{\mathbf{w}}\; \mathbf{w}^\top\Sigma\mathbf{w} \quad \text{s.t.} \quad \mathbf{1}^\top\mathbf{w} = 1$$

### (b) Should a low risk-tolerance investor hold the lower half of the MVF?

**No — this is incorrect.** The minimum variance frontier has two halves split at the GMV point:
- **Upper half** (above GMV): for every σ level, the portfolio on the upper half has *higher* E[Rp] than the corresponding lower-half portfolio.
- **Lower half** (below GMV): these portfolios are *dominated* — they have the same risk as an upper-half portfolio but lower return.

No rational investor — regardless of risk tolerance — should hold a lower-half portfolio. A very risk-averse investor should hold the **GMV portfolio** itself (the leftmost point), not something below it.

### (c) Capital Market Line (CML)

The **CML** is constructed by drawing a **straight line from the risk-free asset** (point (0, Rf) on the σ–E[R] plane) **tangent to the efficient frontier** of risky assets.

- The tangency point gives the **tangency (market) portfolio** M.
- Any point on the CML is a combination of the risk-free asset and M.
- **Slope of the CML = Sharpe Ratio of the tangency portfolio:**
$$\text{Slope} = \frac{E[R_M] - R_f}{\sigma_M}$$
This slope is the maximum Sharpe Ratio achievable and represents the best risk-return trade-off available to any investor combining risky assets with a risk-free asset.


## Question 9 [1 mark] — MVO Inputs & Limitations

### (a) Standard Error of Sample Mean


In [ ]:
# Q9 (a) — Standard error of sample mean
T      = 36          # observations (3 years monthly)
sig_i  = 0.30        # monthly σ = 30%
SE_mu  = sig_i / np.sqrt(T)
print(f"SE(μ̂) = σ / √T = {sig_i} / √{T} = {SE_mu:.4f} ({SE_mu*100:.2f}%)")
print()
print("Implication: With only 36 data points, the standard error of the")
print("estimated monthly mean return is ≈5%, which is HUGE relative to")
print("typical monthly returns of 0.5–1.5%.")
print("→ Historical sample means are extremely noisy estimates of true μ.")
print("→ Relying on them alone leads MVO to dramatically overweight")
print("  assets that happened to perform well in-sample (estimation error).")


### (b) Error Maximisation Property of MVO

MVO is called an **"error maximiser"** because:
- It heavily overweights assets with high estimated μ̂ (which may be estimation noise, not true alpha).
- Small errors in the estimated μ inputs produce **large, unstable** shifts in optimal weights.
- The optimiser treats noise in μ̂ as signal and "maximises" allocations to high-noise assets.

**How ML-predicted returns help:**  
ML models (e.g., gradient boosting, neural networks) use features (momentum, macro variables, fundamental ratios) to produce **shrinkage-type** forecasts of μ that are less extreme than raw sample means. They implicitly or explicitly regularise return estimates, reducing the out-of-sample estimation error — so MVO produces more stable, diversified, and realistic portfolios.

### (c) Number of Parameters in Σ for N assets


In [ ]:
# Q9 (c) — Number of unique parameters in Σ
N = 50
# Σ is a symmetric N×N matrix with 1s on diagonal for correlations,
# but as covariance matrix it has N variances + N(N-1)/2 covariances
n_variances   = N
n_covariances = N * (N - 1) // 2
n_total       = n_variances + n_covariances
print(f"N = {N} assets")
print(f"Variances  (diagonal): {n_variances}")
print(f"Covariances (off-diag, unique): N(N-1)/2 = {N}×{N-1}//2 = {n_covariances}")
print(f"Total unique parameters in Σ: {n_total}")
print()
print("Challenge: With T=36 monthly observations per asset (108 total"),
print("data points for 3 years), estimating 1325 parameters reliably")
print("is statistically ill-posed (far more unknowns than observations).")
print("The sample covariance matrix is often ill-conditioned or singular,")
print("leading to extreme and unstable portfolio weights in MVO.")


---
# §2 Coding Questions — NumPy (Q10–Q12)


## Question 10 [1 mark] — Monthly Returns & Covariance with NumPy

In [ ]:
# ── Q10: Given price array ──────────────────────────────────────────────
prices = np.array([
    [100, 108, 103, 115, 110, 119, 125, 121, 130, 127, 135, 140],  # Stock A
    [200, 195, 210, 205, 220, 215, 225, 230, 222, 235, 240, 238]   # Stock B
])

# (a) Monthly simple returns — vectorised slicing, shape (2, 11)
returns = (prices[:, 1:] - prices[:, :-1]) / prices[:, :-1]
print("=== (a) Monthly Simple Returns (shape:", returns.shape, ") ===")
print("Stock A:", np.round(returns[0], 4))
print("Stock B:", np.round(returns[1], 4))


In [ ]:
# (b) Annualised mean return and standard deviation
monthly_mean = returns.mean(axis=1)
monthly_std  = returns.std(axis=1, ddof=1)   # sample std

ann_mean = monthly_mean * 12
ann_std  = monthly_std  * np.sqrt(12)

print("=== (b) Annualised Statistics ===")
for i, name in enumerate(['Stock A', 'Stock B']):
    print(f"  {name}: Ann. Mean = {ann_mean[i]*100:.2f}%   Ann. Std = {ann_std[i]*100:.2f}%")


In [ ]:
# (c) 2×2 Sample Covariance Matrix using np.cov
# np.cov expects each row = one variable (2 stocks × 11 observations)
cov_mat = np.cov(returns)          # shape (2,2), ddof=1 by default
print("=== (c) 2×2 Covariance Matrix ===")
print(cov_mat)

# Verify off-diagonal: should equal ρ·σA·σB
rho_AB      = np.corrcoef(returns)[0, 1]
sigma_A_mo  = returns[0].std(ddof=1)
sigma_B_mo  = returns[1].std(ddof=1)
manual_cov  = rho_AB * sigma_A_mo * sigma_B_mo

print(f"\nVerification of off-diagonal entry:")
print(f"  np.cov off-diag   = {cov_mat[0,1]:.8f}")
print(f"  ρ·σA·σB           = {manual_cov:.8f}")
print(f"  Match? {np.isclose(cov_mat[0,1], manual_cov)} ✓")
print(f"  Sample correlation ρ_AB = {rho_AB:.4f}")


## Question 11 [1 mark] — Equal-Weight Portfolio & Monte Carlo Simulation

In [ ]:
# ── Q11 Setup: 3-asset parameters from Q6 ───────────────────────────────
mu_3  = np.array([0.15, 0.08, 0.05])
sig_3 = np.array([0.25, 0.12, 0.04])
rho_3 = np.array([[1.0, 0.4, 0.1],
                   [0.4, 1.0, 0.2],
                   [0.1, 0.2, 1.0]])
Sigma_3 = np.diag(sig_3) @ rho_3 @ np.diag(sig_3)

# (a) Equal-weight portfolio
w_eq = np.array([1/3, 1/3, 1/3])

E_Rp_eq  = w_eq @ mu_3                    # w^T μ
var_p_eq = w_eq @ Sigma_3 @ w_eq          # w^T Σ w
sig_p_eq = np.sqrt(var_p_eq)

print("=== (a) Equal-Weight Portfolio ===")
print(f"  E[Rp]  = w^T μ      = {E_Rp_eq*100:.4f}%")
print(f"  σp²    = w^T Σ w    = {var_p_eq:.6f}")
print(f"  σp     =            = {sig_p_eq*100:.4f}%")


In [ ]:
# (b) 10,000 random portfolios using Dirichlet distribution
np.random.seed(42)
N_sim  = 10_000
W_rand = np.random.dirichlet(np.ones(3), N_sim)   # shape (10000, 3), rows sum to 1

# Vectorised: E[Rp] for each portfolio
E_rand = W_rand @ mu_3                             # shape (10000,)

# Vectorised: σp for each portfolio — W @ Σ @ W^T gives covariance matrix,
# but we need per-portfolio variance = diag(W Σ W^T)
# More efficient: var_i = sum_j sum_k w_j * w_k * Σ_jk
# = einsum or (W @ Sigma) * W summed over columns
var_rand = np.sum((W_rand @ Sigma_3) * W_rand, axis=1)   # shape (10000,)
sig_rand = np.sqrt(var_rand)                              # shape (10000,)

print("=== (b) Random Portfolio Simulation ===")
print(f"  E_rand shape: {E_rand.shape}")
print(f"  sig_rand shape: {sig_rand.shape}")
print(f"  E[Rp] range: [{E_rand.min()*100:.2f}%, {E_rand.max()*100:.2f}%]")
print(f"  σp    range: [{sig_rand.min()*100:.2f}%, {sig_rand.max()*100:.2f}%]")


In [ ]:
# (c) Sharpe Ratios — fully vectorised, no loops
Rf_q11 = 0.04
sharpe_rand = (E_rand - Rf_q11) / sig_rand    # shape (10000,)

max_idx    = np.argmax(sharpe_rand)
max_sharpe = sharpe_rand[max_idx]
best_w     = W_rand[max_idx]

print("=== (c) Maximum Sharpe Portfolio (from 10,000 simulations) ===")
print(f"  Max Sharpe Ratio : {max_sharpe:.4f}")
print(f"  Weights          : w1={best_w[0]:.4f}  w2={best_w[1]:.4f}  w3={best_w[2]:.4f}")
print(f"  E[Rp]            : {E_rand[max_idx]*100:.2f}%")
print(f"  σp               : {sig_rand[max_idx]*100:.2f}%")
print(f"  Weights sum to 1?: {np.isclose(best_w.sum(), 1)} ✓")


## Question 12 [1 mark] — Correlation Sensitivity of Portfolio Risk

In [ ]:
# ── Q12 ────────────────────────────────────────────────────────────────
mu1_q12, sig1_q12 = 0.12, 0.20
mu2_q12, sig2_q12 = 0.06, 0.10
w1_q12 = 0.6;  w2_q12 = 0.4

# (a) σp for 200 ρ values in [-1, +1]  — fully vectorised
rho_grid = np.linspace(-1, 1, 200)         # shape (200,)

var_grid = (w1_q12**2 * sig1_q12**2
          + w2_q12**2 * sig2_q12**2
          + 2 * w1_q12 * w2_q12 * rho_grid * sig1_q12 * sig2_q12)
sig_grid = np.sqrt(var_grid)               # shape (200,)

print("=== (a) σp array (first 5 values, ρ starting at -1) ===")
print(np.round(sig_grid[:5], 6))
print(f"Array shape: {sig_grid.shape}  ✓")


In [ ]:
# (b) Minimum σp and corresponding ρ
min_idx    = np.argmin(sig_grid)
rho_at_min = rho_grid[min_idx]
sig_at_min = sig_grid[min_idx]

print(f"=== (b) Minimum Portfolio Risk ===")
print(f"  ρ at minimum σp : {rho_at_min:.4f}")
print(f"  Minimum σp       : {sig_at_min*100:.4f}%")
print(f"  (ρ = -1 gives σp = {sig_grid[0]*100:.4f}% — lowest possible)")


### (c) Analytical Verification — Minimum Always at ρ = −1

**Portfolio variance:** σp²(ρ) = w₁²σ₁² + w₂²σ₂² + 2w₁w₂ρσ₁σ₂

Differentiating with respect to ρ:
$$\frac{d(\sigma_p^2)}{d\rho} = 2w_1 w_2 \sigma_1 \sigma_2$$

Since w₁, w₂, σ₁, σ₂ > 0, this derivative is **strictly positive** for all ρ.

This means σp²(ρ) is a **strictly increasing linear function of ρ**.

Therefore the **minimum** always occurs at the left boundary of the domain, i.e. **ρ = −1**.

**Numerical check:**


In [ ]:
# (c) Numerical verification
d_var_d_rho = 2 * w1_q12 * w2_q12 * sig1_q12 * sig2_q12
print(f"  dσp²/dρ = 2·w1·w2·σ1·σ2 = {d_var_d_rho:.6f}  (> 0 always)")
print(f"  → σp² is strictly increasing in ρ → minimum at ρ = -1 ✓")
print(f"  σp at ρ=-1  : {sig_grid[0]*100:.4f}%")
print(f"  σp at ρ=+1  : {sig_grid[-1]*100:.4f}%  (maximum)")


---
# §3 Coding Questions — Pandas (Q13–Q14)


## Question 13 [1 mark] — Simulated Price DataFrame

In [ ]:
# ── Q13 Setup — EXACTLY as given in the assignment ──────────────────────
import pandas as pd
import numpy as np

np.random.seed(0)
dates       = pd.date_range('2023-01-02', periods=52, freq='W-MON')
mu_weekly   = np.array([0.003, 0.002, 0.001, 0.0015])
sig_weekly  = np.array([0.04,  0.03,  0.02,  0.025])
returns_sim = np.random.normal(mu_weekly, sig_weekly, (52, 4))
prices_sim  = 100 * np.cumprod(1 + returns_sim, axis=0)

df = pd.DataFrame(prices_sim, index=dates, columns=['AAPL', 'MSFT', 'GOOGL', 'AMZN'])

print("Price DataFrame head:")
print(df.head(3))
print(f"\nPrice DataFrame shape: {df.shape}")


In [ ]:
# (a) Weekly simple returns
df_returns = df.pct_change().dropna()
print("=== (a) Weekly Simple Returns — first 3 rows ===")
print(df_returns.head(3).to_string())
print(f"\nShape of returns DataFrame: {df_returns.shape}")


In [ ]:
# (b) .describe() — identify highest mean and highest std
desc = df_returns.describe()
print("=== (b) .describe() on Returns DataFrame ===")
print(desc.to_string())

highest_mean = desc.loc['mean'].idxmax()
highest_std  = desc.loc['std'].idxmax()
print(f"\n→ Highest mean return  : {highest_mean} ({desc.loc['mean', highest_mean]*100:.4f}%/week)")
print(f"→ Highest std deviation: {highest_std}  ({desc.loc['std',  highest_std]*100:.4f}%/week)")


In [ ]:
# (c) Annualised Sharpe Ratio — Pandas only, no loops
Rf_annual  = 0.02
Rf_weekly  = Rf_annual / 52          # weekly risk-free rate

# Annualised mean: weekly mean × 52
# Annualised std:  weekly std  × √52
ann_mean_sr = df_returns.mean() * 52
ann_std_sr  = df_returns.std()  * np.sqrt(52)

sharpe_ann  = (ann_mean_sr - Rf_annual) / ann_std_sr   # Pandas Series, no loop

print("=== (c) Annualised Sharpe Ratios (Rf = 2%) ===")
for ticker in df_returns.columns:
    print(f"  {ticker}: Ann.Mean={ann_mean_sr[ticker]*100:.2f}%  "
          f"Ann.Std={ann_std_sr[ticker]*100:.2f}%  Sharpe={sharpe_ann[ticker]:.4f}")
print(f"\nBest Sharpe: {sharpe_ann.idxmax()} ({sharpe_ann.max():.4f})")


## Question 14 [1 mark] — Correlation Matrix, Equal-Weight Portfolio & Monthly Resampling

In [ ]:
# (a) 4×4 Correlation Matrix
corr_mat = df_returns.corr()
print("=== (a) Correlation Matrix ===")
print(corr_mat.to_string())

# Find the pair with lowest correlation (off-diagonal)
corr_arr = corr_mat.to_numpy().copy()
np.fill_diagonal(corr_arr, np.nan)
corr_val = pd.DataFrame(corr_arr, index=corr_mat.index, columns=corr_mat.columns)
min_pair = corr_val.stack().idxmin()
print(f"\n→ Lowest correlation pair: {min_pair[0]} & {min_pair[1]} = {corr_val.stack().min():.4f}")


In [ ]:
# (b) Equal-weight portfolio return series using .dot()
weights = pd.Series([0.25, 0.25, 0.25, 0.25], index=df_returns.columns)
port_returns = df_returns.dot(weights)    # Pandas Series, 51 weekly values
print("=== (b) Equal-Weight Portfolio Returns (first 5) ===")
print(port_returns.head())
print(f"\nPortfolio return series shape: {port_returns.shape}")
print(f"Weekly mean: {port_returns.mean()*100:.4f}%")


In [ ]:
# (c) Resample to monthly using compound return aggregation
# Monthly compounded return = prod(1 + rw) - 1
monthly_returns = port_returns.resample('ME').apply(lambda x: (1 + x).prod() - 1)

print("=== (c) Monthly Portfolio Returns ===")
print(monthly_returns.to_string())
print(f"\nMonthly mean : {monthly_returns.mean()*100:.4f}%")
print(f"Monthly std  : {monthly_returns.std()*100:.4f}%")
print(f"Number of months: {len(monthly_returns)}")


---
# §4 Coding Questions — Matplotlib (Q15)
## Question 15 [2 marks] — Efficient Frontier & Correlation Sensitivity Plot


In [ ]:
# ── Q15: Full Publication-Quality Figure ─────────────────────────────────
np.random.seed(42)

# ─── Parameters ──────────────────────────────────────────────────────────
mu_15   = np.array([0.15, 0.08, 0.05])
sig_15  = np.array([0.25, 0.12, 0.04])
rho_15  = np.array([[1.0, 0.4, 0.1],
                    [0.4, 1.0, 0.2],
                    [0.1, 0.2, 1.0]])
Sigma_15 = np.diag(sig_15) @ rho_15 @ np.diag(sig_15)
Rf_15   = 0.04
tickers = ['Asset 1', 'Asset 2', 'Asset 3']

# ─── Simulate 20,000 random portfolios ───────────────────────────────────
N_sim15 = 20_000
W15      = np.random.dirichlet(np.ones(3), N_sim15)
E15      = W15 @ mu_15
Var15    = np.sum((W15 @ Sigma_15) * W15, axis=1)
Sig15    = np.sqrt(Var15)
SR15     = (E15 - Rf_15) / Sig15

# ─── Max Sharpe portfolio ─────────────────────────────────────────────────
ms_idx   = np.argmax(SR15)
ms_E     = E15[ms_idx];   ms_S = Sig15[ms_idx]
ms_w     = W15[ms_idx];   ms_SR = SR15[ms_idx]

# ─── Subplot 2 params (Q12) ───────────────────────────────────────────────
mu1_p, sig1_p = 0.12, 0.20
mu2_p, sig2_p = 0.06, 0.10
w1_p, w2_p    = 0.60, 0.40
rho_p         = np.linspace(-1, 1, 200)
var_p         = (w1_p**2*sig1_p**2 + w2_p**2*sig2_p**2
               + 2*w1_p*w2_p*rho_p*sig1_p*sig2_p)
sig_p_arr     = np.sqrt(var_p)
w_avg_p       = w1_p*sig1_p + w2_p*sig2_p

# ─── Figure ───────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Portfolio Theory — Week 1 Visualisations',
             fontsize=15, fontweight='bold', y=1.01)

# ── SUBPLOT 1: Efficient Frontier ─────────────────────────────────────────
norm  = Normalize(vmin=SR15.min(), vmax=SR15.max())
cmap  = plt.get_cmap('viridis')
sc    = ax1.scatter(Sig15 * 100, E15 * 100,
                    c=SR15, cmap='viridis', alpha=0.4, s=5, linewidths=0)

# Max Sharpe star
ax1.scatter(ms_S * 100, ms_E * 100,
            marker='*', s=350, color='gold', edgecolors='darkgoldenrod',
            linewidths=1.2, zorder=5, label=f'Max Sharpe ({ms_SR:.2f})')

# Individual assets
asset_colors = ['#e74c3c', '#2980b9', '#27ae60']
for i, (ticker, color) in enumerate(zip(tickers, asset_colors)):
    ax1.scatter(sig_15[i] * 100, mu_15[i] * 100,
                marker='o', s=180, color=color, zorder=6,
                edgecolors='black', linewidths=1)
    ax1.annotate(ticker,
                 xy=(sig_15[i]*100, mu_15[i]*100),
                 xytext=(6, 4), textcoords='offset points',
                 fontsize=9, fontweight='bold', color=color)

# Colorbar
cbar = fig.colorbar(sc, ax=ax1, pad=0.02)
cbar.set_label('Sharpe Ratio', fontsize=10)

# Axes
ax1.xaxis.set_major_formatter(mticker.PercentFormatter())
ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
ax1.set_xlabel('Portfolio Risk  σp (%)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Expected Return  E[Rp] (%)', fontsize=11, fontweight='bold')
ax1.set_title('Efficient Frontier — 20,000 Random Portfolios', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(True, alpha=0.3)

# ── SUBPLOT 2: Correlation Sensitivity ───────────────────────────────────
ax2.plot(rho_p, sig_p_arr * 100, color='steelblue', linewidth=2.5,
         label=r'$sigma_p(rho)$')

# Horizontal dashed line at weighted-average risk
ax2.axhline(w_avg_p * 100, color='crimson', linestyle='--', linewidth=1.8,
            label=f'Weighted Avg. Risk ({w_avg_p*100:.2f}%)')

# Shade diversification benefit region
ax2.fill_between(rho_p, sig_p_arr * 100, w_avg_p * 100,
                 where=(sig_p_arr * 100 < w_avg_p * 100),
                 color='lightgreen', alpha=0.5, label='Diversification Benefit')

# Mark ρ = -1 minimum
ax2.scatter([-1], [sig_p_arr[0] * 100], color='darkgreen', s=120,
            zorder=5, label=f'Min σp at ρ=-1 ({sig_p_arr[0]*100:.2f}%)')

ax2.set_xlabel('Correlation  ρ₁₂', fontsize=11, fontweight='bold')
ax2.set_ylabel('Portfolio Risk  σp (%)', fontsize=11, fontweight='bold')
ax2.set_title('Correlation Sensitivity of Portfolio Risk(2-Asset: μ₁=12%, σ₁=20%, μ₂=6%, σ₂=10%, w₁=0.6)',
              fontsize=11, fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.set_xlim(-1.05, 1.05)
ax2.legend(fontsize=9, loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week1_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as 'week1_plots.png' ✓")
print(f"Max Sharpe = {ms_SR:.4f} at weights: {np.round(ms_w, 4)}")


---
## ✅ Assignment Complete — Summary

| Question | Marks | Topic |
|----------|-------|-------|
| Q1 | 2 | Simple returns, log returns, approximation |
| Q2 | 1 | Portfolio expected return, linearity |
| Q3 | 2 | Derivation of portfolio variance formula |
| Q4 | 1 | Two-asset σp at ρ = ±1, 0; zero-risk weight |
| Q5 | 1 | Diversification intuition (<150 words) |
| Q6 | 1 | 3×3 covariance matrix construction |
| Q7 | 1 | Sharpe ratios; ranking; rational investor |
| Q8 | 1 | GMV portfolio; efficient frontier; CML |
| Q9 | 1 | MVO limitations; SE of mean; error maximisation |
| Q10 | 1 | NumPy: returns, annualised stats, covariance |
| Q11 | 1 | NumPy: equal-weight; Monte Carlo; max Sharpe |
| Q12 | 1 | NumPy: σp vs ρ; minimum; analytical proof |
| Q13 | 1 | Pandas: pct_change, describe, Sharpe |
| Q14 | 1 | Pandas: correlation, dot product, resample |
| Q15 | 2 | Matplotlib: efficient frontier + sensitivity plot |
| **Total** | **17** | |

> Reference: Blanchard & Johnson, *Macroeconomics*; Markowitz (1952), *Portfolio Selection*
